# 01 - Data Understanding and Baseline Cleaning

This notebook loads the cleaned CMS-style dataset, validates the expected columns, standardizes numeric/text fields, and creates a clean base table for modeling.

## Expected input columns
- `provider_id`
- `provider_name`
- `provider_state`
- `drg_definition`
- `average_covered_charges`
- `average_total_payments`
- `Geography`
- `Service type (ER, imaging, etc.)`
- `Billed amount`
- `Medicare payment`

## Output
- `data/base_cleaned.parquet`
- summary tables for presentation

In [ ]:

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = "data/cleaned_cms.csv"   # change this if needed
assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

In [ ]:

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:

expected_cols = [
    "provider_id", "provider_name", "provider_state", "drg_definition",
    "average_covered_charges", "average_total_payments", "Geography",
    "Service type (ER, imaging, etc.)", "Billed amount", "Medicare payment"
]

missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[expected_cols].copy()
df.info()

In [ ]:

def clean_money(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.replace(r"[$,]", "", regex=True)
              .str.strip()
              .replace({"": np.nan, "nan": np.nan, "None": np.nan})
              .astype(float)
    )

money_cols = [
    "average_covered_charges",
    "average_total_payments",
    "Billed amount",
    "Medicare payment",
]

for col in money_cols:
    df[col] = clean_money(df[col])

text_cols = ["provider_name", "provider_state", "drg_definition", "Geography", "Service type (ER, imaging, etc.)"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

df["provider_id"] = df["provider_id"].astype(str).str.strip()

# basic standardization
df["provider_state"] = df["provider_state"].str.upper().str[:2]
df["service_type"] = df["Service type (ER, imaging, etc.)"].str.lower()
df["drg_text"] = df["drg_definition"].str.lower()

# deduplicate
before = len(df)
df = df.drop_duplicates()
print("Dropped duplicates:", before - len(df))
df.head()

In [ ]:

# Missingness summary
missing_summary = (
    df.isna().mean().sort_values(ascending=False).rename("missing_pct").reset_index()
    .rename(columns={"index": "column"})
)
missing_summary

In [ ]:

# Simple imputation for a modeling-friendly base table
for col in money_cols:
    df[col] = df.groupby("provider_state")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())

for col in ["provider_name", "provider_state", "Geography", "service_type", "drg_text"]:
    df[col] = df[col].fillna("unknown")

df[money_cols].describe().T

In [ ]:

# Quick presentation visuals
state_counts = df["provider_state"].value_counts().head(15)
plt.figure(figsize=(10, 5))
state_counts.plot(kind="bar")
plt.title("Top 15 States by Record Count")
plt.xlabel("State")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:

service_counts = df["service_type"].value_counts().head(12)
plt.figure(figsize=(10, 5))
service_counts.plot(kind="bar")
plt.title("Top Service Categories")
plt.xlabel("Service Type")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:

os.makedirs("data", exist_ok=True)
df.to_parquet("data/base_cleaned.parquet", index=False)
print("Saved to data/base_cleaned.parquet")